# OpenQuake

[**OpenQuake Engine**](https://www.globalquakemodel.org/product/openquake-engine) ([Silva *et al*., 2014](https://link.springer.com/article/10.1007/s11069-013-0618-x)) is an open-source software for seismic hazard and risk assessment developed by the [**Global Earthquake Model (GEM) foundation**](https://www.globalquakemodel.org). The engine is a combination of different libraries allowing:

* Compute `Ground Motion fields`.
* Calculate `Seismic Hazard curves`.
* Calculate `Seismic Hazard disaggregation`.
* Describe `exposure, vulnerability, and fragility curves`.
* Estimate `losses` due to seismic events.


<img src="https://pypi-camo.freetls.fastly.net/9c27003502eaab768dd8c960ed1ba90c55190555/68747470733a2f2f7261772e67697468756275736572636f6e74656e742e636f6d2f67656d2f6f712d696e6672617374727563747572652f6d61737465722f6c6f676f732f6f712d6c6f676f2e706e67" width="500" height="200" />

*Image credit: [Global Earthquake Model](https://www.globalquakemodel.org/product/openquake-engine)*

**OpenQuake Engine** can be accessed using the command line, the OpenQuake Web UI, the OpenQuake QGIS plugin, or by importing its libraries directly into Python scripts. For a complete tutorial using the OpenQuake Web UI, visit https://www.training.openquake.org/.

# 0. Objectives

* Use the **USGS ComCat** database to retrieve source parameters required to implement Ground Motion Models (GMMs) in Engineering Practice.
* Calculate **distance metrics** required in GMMs for different rupture models using **OpenQuake engine**.
* Estimate ground motion Intensity Measures (IM) at given locations and their corresponding uncertainty using GMMs through the **OpenQuake engine** hazard library.
* Utilize **Next Generation Attanuation (NGA) flatfiles** to retrieve ground motion IM and metadata for source, path, and site conditions of specific events.
* Examine the performance of GMMs by comparing model predictions with observed ground motions.

# 1. Installing openquake.engine

OpenQuake Engine is available on Windows, macOS, and Linux. The current Long Term Support (LTS) release of **OpenQuake engine** (v3.23) requires `Python 3.11` or older (Python versions 3.13 and newer are not supported on this branch) and it can be installed using pip, conda or a [universal installer](https://docs.openquake.org/oq-engine/3.23/manual/getting-started/installation-instructions/index.html). For this workshop, we will require also **pyGMT** as a visualization tool. To handle the complex underlying, we recomend creating a `venv` using **conda**:

<div style="border: 1px solid #e0e0e0; border-radius: 2px; padding: 5px; font-family: sans-serif; background-color: #fafafa;">
  <div style="background-color: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 4px; padding: 12px; font-family: monospace; position: relative; line-height: 1.6;">
    <code style="color: #4466ff;">conda create --prefix ./.venv --channel conda-forge python=3.10 "pygmt<=0.17.0"</code><br>
    <code style="color: #4466ff;">conda activate ./.venv</code><br>
    <code style="color: #4466ff;">conda install conda-forge::openquake.engine==3.23.3</code><br>
  </div>
</div>

We will additionally require the following dependencies:

<div style="border: 1px solid #e0e0e0; border-radius: 2px; padding: 5px; font-family: sans-serif; background-color: #fafafa;">
  <div style="background-color: #f5f5f5; border: 1px solid #e0e0e0; border-radius: 4px; padding: 12px; font-family: monospace; position: relative; line-height: 1.6;">
    <code style="color: #4466ff;">conda install -c conda-forge ipykernel openpyxl geopandas fiona scipy
</code><br>
  </div>
</div>

In [ ]:
import pygmt
import pandas as pd
pygmt.show_versions()

# 2. Event: 1999 Chi-Chi, Taiwan Earthquake

In this workshop, we will analyze information about the [`1999 Chi-Chi` earthquake](https://earthquake.usgs.gov/earthquakes/eventpage/usp0009eq0/executive). The $M_W$ 7.6 event ruptured an approximately 100-km segment of Chelengpu thrust fault and it was classified as an **active crustal** earthquake [(Garcia *et al*, 2012)](https://pubs.geoscienceworld.org/ssa/bssa/article/102/1/185/349607/A-Global-Earthquake-Discrimination-Scheme-to).

<img src="https://earthquake.usgs.gov/product/shakemap/usp0009eq0/atlas/1594169383237/download/intensity.jpg" width="300" height="400" />

The `1999 Chi-Chi` earthquake was the largest on land earhtquake recorded in Taiwan in the 20th century [(Shin and Teng, 2001)](https://pubs.geoscienceworld.org/ssa/bssa/article-abstract/91/5/895/102886/An-Overview-of-the-1999-Chi-Chi-Taiwan-Earthquake?redirectedFrom=fulltext). It caused considerable damage with at least 2,297 people killed, 8,700 injured, and a total economic loss estimated at 14 billion U.S. dollars. At the time, the `1999 Chi-Chi` earthquake generated the largest set of strong ground motion records ever captured from a single event [(Lee *et al.*, 2001)](https://pubs.geoscienceworld.org/ssa/bssa/article-abstract/91/5/1370/102942/CWB-Free-Field-Strong-Motion-Data-from-the-21?redirectedFrom=fulltext), representing a landmark in earthquake engineering history.

<table>
  <tr>
    <td><img src="https://upload.wikimedia.org/wikipedia/commons/9/9c/990921중앙119구조본부_대만_921_대지진_출동1.jpg?utm_source=en.wikipedia.org&utm_campaign=index&utm_content=original" alt="Image 1" width="200"><br>
    <small style="color: #666;">Image credit: <a href="https://en.wikipedia.org/wiki/1999_Jiji_earthquake#/media/File:990921중앙119구조본부_대만_921_대지진_출동1.jpg" target="_blank">Choi Kwang-mo</a></small> </td>
    <td><img src="https://www.eas.slu.edu/eqc/eqc_photos/1999TaiwanEQ/images/001b.jpg" alt="Image 2" width="400"><br> 
    <small style="color: #666;">Image credit: <a href="https://www.eas.slu.edu/eqc/eqc_photos/1999TaiwanEQ/001b.html" target="_blank">Prof. Robert Herrmann</a></small> </td>
    <td><img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRy8P_7_JFrSvQmqWfH9yah3Gg8lS9K_AtCK8KNoY4XkVSmTFFGMhguM4M&s=10" alt="Image 2" width="200"><br>    
    <small style="color: #666;">Image credit: <a href="https://www.facebook.com/APImages/posts/20-years-ago-today-the-chi-chi-earthquake-killed-more-than-2400-people-in-centra/10156746818453865/" target="_blank">Associated Press</a></small></td>
  </tr>
  <tr>
    <td><img src="https://research.engineering.ucdavis.edu/gpa/wp-content/uploads/sites/43/2015/02/Rupture-in-track-field.jpg" alt="Image 3" width="250"><br>
    <small style="color: #666;">Image credit: <a href="https://research.engineering.ucdavis.edu/gpa/earthquake-hazards/surface-rupture-taiwan/" target="_blank">Prof. Ross Boulanger</a></small></td>
    <td><img src="https://cdn.britannica.com/76/139776-050-7256BFE8/Chi-chi-Temple-earthquake-Taiwan-1999.jpg" alt="Image 4" width="400"><br>
    <small style="color: #666;">Image credit: <a href="https://www.britannica.com/event/Taiwan-earthquake-of-1999" target="_blank">Encyclopedia Britannica</a></small> </td>
    <td><img src="https://research.engineering.ucdavis.edu/gpa/wp-content/uploads/sites/43/2015/02/Bridge-and-waterfall.jpg" alt="Image 3" width="260"><br>
    <small style="color: #666;">Image credit: <a href="https://research.engineering.ucdavis.edu/gpa/earthquake-hazards/surface-rupture-taiwan/" target="_blank">Prof. Ross Boulanger</a></small> </td>
  </tr>
</table>

## 2.1. Characteristics of the event

The main characteristics of the `1999 Chi-Chi` earthquake were retrieved from [USGS Earthquake Catalog (ComCat)](https://earthquake.usgs.gov/earthquakes/search/). The data is located at **data/Chi_Chi_event.csv**, and it can be loaded as a `pandas.DataFrame`:

In [ ]:
event = pd.read_csv("data/Chi_Chi_event.csv")
event.head()

We can use this information to visualize the Focal Mechanism using **`pyGMT`**:

In [ ]:
fig = pygmt.Figure()
region = [119.5, 122.5, 21.5, 25.5]
proj = "T121/23.5/10c"

grid = pygmt.datasets.load_earth_relief(resolution="15s", region=region)
pygmt.makecpt(cmap="geo", series=[-8000, 7000, 500], continuous=True, output="data/eleva.cpt")

fig.grdimage(grid=grid, region=region, projection=proj, cmap="data/eleva.cpt")

with pygmt.config(MAP_GRID_PEN="0.1p,black"):
    fig.basemap(frame=["WSne", "a2f1g1"])

fig.meca(spec=event.iloc[0,0:].to_dict(), scale="0.5c+f10p", labelbox="white@30", offset="+p1p,black+s0.15c")

fig.show()

## 2.2. Strong Ground Motion Stations: NGA-West2 Database

Considering its significance and the high quality of the strong-motion records, the `1999 Chi-Chi` mainshock and five major aftershocks were included in the `NGA-West2 database` [(Ancheta *et al.*, 2014)](https://journals.sagepub.com/doi/10.1193/070913EQS197M). In addition to the processed time series available at the [online PEER Ground Motion Database](https://ngawest2.berkeley.edu/), NGA-West2 provided several [`flatfiles`](https://peer.berkeley.edu/research/databases/databases). Flatfiles are spreadsheet-like databases (`.xlsx` files) that compile the `source`, `path`, and `site` metadata and ground-motion (GM) Intensity Measures (`IMs`) used to develop the Ground Motion Models (GMMs).  

The **NGA-West2 flatfile** metadata tables include:

* `Source`: event time, **moment magnitude** ($M_w$), hypocenter location, **focal mechanism**, and finite fault geometry (**strike**, **dip**, **length**, **width**, etc.).
* `Finite Fault model`: plane (or series of planes) that describes the earthquake geometry
* `Path`: site-source path parameters, including epicentral, hypocentral, and Joyner-Boore distances, among others.
* `Site`: station coordinates, **time-averaged shear wave velocity in the upper 30 m** ($V_{S30}$), proxies for $V_{S30}$ estimation (*e.g.,* topographic slope), several site classifications (*e.g.,* NHERP), and basin depth parameters.
* `Signal Processing`: instrument models, type of filters and their corner- frequencies, usable frequencies, etc.
* `IMs`: **PGA**, **PGV**, **PGD**, and pseudo-spectral accelerations (**PSA**) at **111 periods** from **0.01 to 20 s** for **11 damping ratios** from **0.5 to 30 %**. For the horizontal components, NGA-West2 reports the `RotD50` spectra.

The **NGA-West2 flatfile** for horizontal components and **0.5% damping ratio** is located at **data/Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx**. Note that the naming convention uses "d###" format to indicate the damping in tenths. This database can be loaded as a `pandas.DataFrame` using the function `pandas.read_excel()`:

In [ ]:
df_NGAW2 = pd.read_excel("data/Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx")
print(df_NGAW2.head())

The complete list of variables included in the flatfile is:

In [ ]:
print(df_NGAW2.columns.tolist())

We can select the GMs associated with the `1999 Chi-Chi` earthquake by filtering based on `Earthquake Name`:

In [ ]:
df_chichi = df_NGAW2[df_NGAW2['Earthquake Name']=="Chi-Chi, Taiwan"]
print("Number of stations:", len(df_chichi))
df_chichi.head()

The database contains 420 stations distributed across Taiwan:

In [ ]:
fig = pygmt.Figure()
region = [119.5, 122.5, 21.5, 25.5]
proj = "T121/23.5/10c"

grid = pygmt.datasets.load_earth_relief(resolution="15s", region=region)
pygmt.makecpt(cmap="geo", series=[-8000, 7000, 500], continuous=True, output="data/eleva.cpt")

fig.grdimage(grid=grid, region=region, projection=proj, cmap="data/eleva.cpt")

with pygmt.config(MAP_GRID_PEN="0.1p,black"):
    fig.basemap(frame=["WSne", "a2f1g1"])

fig.meca(spec=event.iloc[0,0:].to_dict(), scale="0.5c+f10p", labelbox="white@30", offset="+p1p,black+s0.15c")
fig.plot(x=df_chichi["Station Longitude"], y=df_chichi["Station Latitude"], style="c0.15c",  fill="white", pen="0.5p,black")

fig.show()

In this workshop, we will compare the IMs reported in the flatfile to two NGA-West2 GMMs: `BSSA14` [(Boore *et al.*, 2014)](https://journals.sagepub.com/doi/10.1193/070113EQS184M) and `CY14` [(Chiou and Youngs, 2014)](https://journals.sagepub.com/doi/10.1193/072813EQS219M).

# 3. Ground Motion Models (GMMs): NGA-West2

## 3.1. Mathematical representation

The NGA-West2 GMMs predict the distribution of PGA, PGV, and 5%-damped PSA for shallow crustal earthquakes in active tectonic regions. These empirical GMMs assume that the IMs follow a `lognormal` distribution:

$$\log\left(IM \right) \sim \mathcal{N}\left(\mu_{\log IM}, \sigma^2_{\log IM}\right)$$

where $\mu_{\log IM}$ and $\sigma_{\log IM}$ are the mean and standard deviation. Thus, we can write:

$$\log\left(IM \right) = \mu_{\log IM} + \sigma_{\log IM} \cdot \varepsilon$$

where $\varepsilon$ is a standard normal random variable. 

Both $\mu_{\log IM}$ and $\sigma_{\log IM} $ are explicitly defined as functions that depend on source ($M$), source ($R$), and site ($S$) parameters, *i.e.*:

$$\mu_{\log IM_{ij}} = f_{M,R,S}(M_{i}, R_{ij}, S_{ij}) $$

$$\sigma_{\log IM_{ij}} = g_{M,R,S}(M_{i}, R_{ij}, S_{ij})$$

where $IM_{ij}$ is the intensity measure due to earthquake rupture *i* at site *j*, and $f_{M,R,S}(\cdot)$ and $g_{M,R,S}(\cdot)$ are the corresponding functional forms for the mean and standard deviation, respectively. 

## 3.2. Input parameters

The functional forms of modern GMMs are complex and consider multiple predictors and coefficients (either empirically determined or theoretically constrained). For example, `CY14` uses **11 predictors** and **45 coefficients**. The list of predictors (input parameters) for the `BSSA14` and `CY14` GMMs is presented below:


| **Parameter**         | **Symbol**  | **Units**  | **BSSA14** | **CY14** | 
| :-------              | :------:    | :------:   |  :------:  | :------: | 
| **Source parameters** |         
| Moment magnitude           |  $M_W$      | -     | <input type="checkbox" checked disabled>  | <input type="checkbox" checked disabled> | 
| Depth-to-top of rupture    |  $Z_{tor}$  | km    |  -                                        | <input type="checkbox" checked disabled> | 
| Fault dip                  |  $\delta$   | °     |  -                                        | <input type="checkbox" checked disabled> | 
| Rake (to define style of faulting)   | $\lambda$ | °  |  <input type="checkbox" checked disabled> | <input type="checkbox" checked disabled> | 
| **Path parameters**   |
| Closest distance to rupture plane <br> (rupture distance)                                       | $R_{rup}$ | km   | - | <input type="checkbox" checked disabled> | 
| Horizontal distance to surface<br> projection of the rupture<br> (Joyner-Boore distance)        | $R_{JB}$  | km   | <input type="checkbox" checked disabled> | <input type="checkbox" checked disabled> | 
| Horizontal distance to edge<br> of the rupture measured perpendicular <br>to the fault strike   | $R_{X}$   | km   | - | <input type="checkbox" checked disabled> | 
| Hanging-wall flag <br> (1 for $R_{X} \geq 0$, 0 for $R_{X}<0$)                                  | $F_{HW}$  | -    | - | <input type="checkbox" checked disabled> | 
| **Site parameters** |
| Travel-time averaged shear-wave<br> velocity of the top 30 m         | $V_{S30}$ |  m/s | <input type="checkbox" checked disabled> | <input type="checkbox" checked disabled> |
| Depth to shear-wave velocity of 1.0 km/s                             | $Z_{1.0}$ |  m   |  -                                       | <input type="checkbox" checked disabled> | 
| PGA/PGA/PSA on reference condition<br> (for nonlinear site response) | $y_{ref}$ |  g   | <input type="checkbox" checked disabled> | <input type="checkbox" checked disabled> |
| Reference condition ($V_{S30}$) | - |  m/s | 760 | 1130 | 


The derivation of some of the source and path parameters used as predictor variables requires a `Finite Fault Model (FFM)` to describe the earthquake source geometry. This includes the three different source-to-site distance measures (**$R_{rup}$, $R_{JB}$, and $R_X$**). The vertical cross-section below compares these metrics for a hypothetical site with the more widely known hypocentral (**$R_{hypo}$**) and epicentral (**$R_{epi}$**) distances. 

<table>
  <tbody>
    <tr>
      <td>
        <img src="fig/Path1.jpg" alt="Image 1" width="500"><br>
        <small style="color: #666;">Adapted from <a href="https://journals.sagepub.com/doi/full/10.1193/1.3650372" target="_blank">Kaklamanos et al. (2011)</a></small>
      </td>
      <td style="vertical-align: top;">
        <img src="fig/Path2.jpg" alt="Image 2" width="500"><br>
        <small style="color: #666;">Adapted from <a href="https://journals.sagepub.com/doi/full/10.1193/1.3650372" target="_blank">Kaklamanos et al. (2011)</a></small>
      </td>
    </tr>
  </tbody>
</table>

Note that all GMMs have specific applicability ranges. The recommended limits for `BSSA14` and `CY14`models are:

| **Parameter**         | **Symbol**  | **Units**  | **BSSA14** | **CY14** | 
| :-------              | :------:    | :------:   |  :------:  | :------: | 
| **Applicability** |
| Magintude | $M_{W}$ | - | Strike-slip and reverse: 3 - 8.5<br> Normal: 3 - 7 | Strike-slip: 3.5 - 8.5<br> Reverse and normal: 3.5 - 8 |
| Distance | - | km | $R_{JB} \leq 400$ | $R_{rup} \leq 300$ |
| Site conditions | $V_{S30}$ | m/s | 150 - 1500 | 180 - 1500 |

# 4. Source parameters

`Finite fault models (FFMs)` describe earthquake source geometry using a `plane` or `series of planes` (multi-fault rupture) in the Earth's crust. Fault geometry can be obtained from field observations, inversion of waveform and geodetic data, or aftershock distribution. Different entities and researchers can propose multiple rupture models for the same event (*e.g.,* [Holden 2011](https://pubs.geoscienceworld.org/ssa/srl/article-standard/82/6/783/143877/Kinematic-Source-Model-of-the-22-February-2011-Mw) and [Beaven *et al.* 2011](https://bulletin.nzsee.org.nz/article/view/250) for the 2011 $M_w$ 6.2 Christchurch earthquake), but many events lack published finite fault models. When a FFM is not available,  researchers develop approximate simulated finite faults based on hypocenter location, magnitude, and focal mechanisms. For further details about the modeling process, check [Chiou and Youngs (2006)](https://apps.peer.berkeley.edu/research/lifelines/nga_docs/jul_10_06/Chiou_Youngs_NGA_2006.pdf).

For the `1999 Chi-Chi` earthquake, we will work with the **simplified version** used by USGS to create the event's [ShakeMap](https://earthquake.usgs.gov/earthquakes/eventpage/usp0009eq0/shakemap/intensity). This is an adaptation of the Finite Fault Model developed by NGA-West2, and it is represented by a series of 3D points that form a closed irregular polygon. This data is provided in `JSON` format and can be found at **data/rupture_chichi.json**:


In [ ]:
import json
import geopandas as gpd
import io

# Reading as a GeoDataFrame to visualize the rupture geometry
rupture = gpd.read_file("data/rupture_chichi.json")

# Exploring the rupture geometry
with open('data/rupture_chichi.json', 'r') as file:
    rupture_json = json.load(file)

rupture_json



USGS also provides a more comprehensive (and complex) multi-segment rupture geometry (2 or more contiguous planar quadrilaterals) on the `1999 Chi-Chi` [Finite Fault webpage](https://earthquake.usgs.gov/earthquakes/eventpage/usp0009eq0/finite-fault). This information is provided as a series of **polygons** in `GEOJSON` format, and it is available at **data/FFM_chichi.geojson**: 

In [ ]:
fault_gdf = gpd.read_file("data/FFM_chichi.geojson")
gmt_input_data = fault_gdf[["fill", "geometry"]]

fault_gdf

We can visualize and compare the two `FFMs` using `pyGMT`:

In [ ]:
fig = pygmt.Figure()

region = [119.5, 122.5, 21.5, 25.5]
proj = "T121/23.5/?"

grid = pygmt.datasets.load_earth_relief(resolution="15s", region=region)
pygmt.makecpt(cmap="geo", series=[-8000, 7000, 500], continuous=True, output="data/eleva.cpt")
pygmt.makecpt(cmap = "matlab/hot", reverse = True, series=[0, 6, 0.25], continuous=True, output="data/slip.cpt")

with fig.subplot(nrows=1, ncols=2, figsize=["20c", "15c"], margins = "1c"):

    with fig.set_panel(panel=0):

        fig.grdimage(grid=grid, region=region, projection=proj, cmap="data/eleva.cpt")
        fig.basemap(frame=["a2f1g1", '+tUSGS FFM'])

        fig.plot(
            data=fault_gdf,
            cmap="data/slip.cpt",
            fill="+z",               
            aspatial="Z=slip",        
            pen="0.1p,black"
        )

        fig.meca(spec=event.iloc[0,0:].to_dict(), scale="0.5c+f10p", labelbox="white@30", offset="+p1p,black+s0.15c")
        fig.plot(x=df_chichi["Station Longitude"], y=df_chichi["Station Latitude"], style="c0.15c", fill="white", pen="0.5p,black")

        fig.colorbar(cmap="data/slip.cpt", frame=['x+lSlip (cm)', 'y'], position="jBL+w4c/0.3c+o0.2c/0.85c+h")

    with fig.set_panel(panel=1):

        fig.grdimage(grid=grid, region=region, projection=proj, cmap="data/eleva.cpt")
        fig.basemap(frame=["a2f1g1", '+tNGA-West2 FFM'])

        fig.plot(
                data=rupture,
                pen="2p,black", 
                fill = "grey@40"
        )

        fig.meca(spec=event.iloc[0,0:].to_dict(), scale="0.5c+f10p", labelbox="white@30", offset="+p1p,black+s0.15c")
        fig.plot(x=df_chichi["Station Longitude"], y=df_chichi["Station Latitude"], style="c0.15c", fill="white", pen="0.5p,black")

fig.show(width=1200)

# 5. Path parameters

We can calculate the path parameters for each of the seismic stations using the `openquake.hazardlib`. In **OpenQuake Engine**, each 3D geographical location is represented by a `Point` object, defined by its:

* `longitude` (°).
* `latitude` (°).
* `depth` (km). Default value of 0, indicating earth surface. 

`Points` are the library's fundamental building block. They are used to construct higher-level objects, including `Mesh` (collection of `Points`) and rupture geometries.

Rupture geometries are divided into different categories, including:

* **Planar surface** - `PlanarSurface()`. 
    * Used for simple, perfectly `flat` and `straight` faults with `constant dip` (rigid horizontally and vertically).
    * Defined by the `four 3D points` of their corners.
* **Simple Fault Surface** - `SimpleFaultSurface()`. 
    * Used to define `curve/jagged` faults with `constant dip` (flexible horizontally, but rigid vertically).
    * Defined by  a `top_trace` and a `bottom_trace`.
* **Complex Fault Surface** - `ComplexFaultSurface()`.
    * Used to define `curve/jagged` faults with `varying dip` (flexible horizontally and vertically).
    * Defined by `multiple contour lines`.
* **Multi-Surface** - `MultiSurface()`. 
    * Used for **multi-fault ruptures** that can be **completely disconnected**.
    * Defined by a Python `list` containing multiple individual surface objects.


<table>
   <tbody>
     <!-- Existing First Row -->
     <tr>
       <td>
         <img src="fig/PlanarSurface.jpg" alt="Image 1" height="200"><br>
         <small style="color: #666;"> Planar Surface</small>
       </td>
       <td style="vertical-align: top;">
         <img src="fig/SimpleFaultSurface.jpg" alt="Image 2" height="200"><br>
         <small style="color: #666;">Simple Fault Surface</small>
       </td>
     </tr>
     <tr>
       <td>
         <img src="fig/ComplexSurface.jpg" alt="Image 3" height="200"><br>
         <small style="color: #666;">Complex Fault Surface</small>
       </td>
       <td style="vertical-align: top;">
         <img src="fig/Multisurface.jpg" alt="Image 4" height="200"><br>
         <small style="color: #666;">Multisurface</small>
       </td>
     </tr>
   </tbody>
 </table>



Given the complexity of the `1999 Chi-Chi` rupture model, we will use the `SimpleFaultSurface()` function. Additionally, we will learn how to define a `SimpleSurface()` for simpler rupture models.

## 5.1. Simple fault: 1999 $M_{W}7.6$ Chi-Chi Earthquake 

We will start by creating a `Mesh()` object containing `Point()` objects for all our seismic stations:

In [ ]:
import numpy as np
from openquake.hazardlib.geo import Point, Mesh

# Reading points
df_stations = pd.DataFrame()
df_stations['station'] = df_chichi['Station Name'].values
df_stations['lon'] = df_chichi['Station Longitude'].values
df_stations['lat'] = df_chichi['Station Latitude'].values
depth_site = np.zeros_like(df_stations['lon'] , dtype=float)

site_mesh = Mesh(
    np.array(df_stations['lon'].values), 
    np.array(df_stations['lat'].values), 
    np.array(depth_site)
)

To determine the $R_{hypo}$ and $R_{epi}$, we need to declare a `Point()` object with the **longitude**, **latitude**, and **depth** of the event hypocenter:

In [ ]:
lon_epi = event['longitude'].values
lat_epi = event['latitude'].values
depth_epi = event['depth'].values
eq_point = Point(lon_epi, lat_epi, depth_epi)

To compute the distance between the hypocenter ($R_{hypo}$) and each point of our station `Mesh()`, we can use the `distance_to_mesh()` method of the `Point()` object with the attribute `with_depths` set to `True`. If we are interested in the distance between the projections of the points ($R_{epi}$), we assign the attribute `with_depths` set to `False`:

In [ ]:
df_stations['r_epi'] = eq_point.distance_to_mesh(site_mesh, with_depths=False)
df_stations['r_hypo'] = eq_point.distance_to_mesh(site_mesh, with_depths=True)
df_stations.head()

The `SimpleFaultSurface()` function expects a `top_trace` and a `bottom_trace`. The edges must be ordered sequentially from the top (fault trace, in blue) to the bottom edge (base, in orange). The points in each edge must be ordered sequentially, following the same strike direction. The fault surface must dip to the right relative to the direction you are moving along the strike line.

<img src="fig/def_SimpleFaultSurface.jpg" alt="Image 2" width="500"><br>

Our `GEOJSON` polygons form a closed loop, where the lower boundary run backwards relative to the top points:

In [ ]:
rupture_json

Therefore, we have to divide our coordinates and sort them in the desired order:

In [ ]:
from openquake.hazardlib.geo.mesh import RectangularMesh
from openquake.hazardlib.geo.surface import PlanarSurface, SimpleFaultSurface

polygon_coordinates = rupture_json['features'][0]['geometry']['coordinates'][0][0]

# Remove negative depths from the rupture coordinates - Drop the last coordinate which is a duplicate of the first
clean_coords = [(lon, lat, abs(depth)) for lon, lat, depth in polygon_coordinates[:-1]]

# Get the midpoint index of the clean coordinates list
midpoint = len(clean_coords) // 2 

# Divide the coordinates into top and bottom traces
top_vertex = clean_coords[:midpoint]
bottom_vertex = clean_coords[midpoint:]

# Correct the strike of the bottom trace
bottom_vertex.reverse()

# Construct the array for the RectangularMesh
lons = np.array([[pt[0] for pt in top_vertex], [pt[0] for pt in bottom_vertex]])
lats = np.array([[pt[1] for pt in top_vertex], [pt[1] for pt in bottom_vertex]])
depths = np.array([[pt[2] for pt in top_vertex], [pt[2] for pt in bottom_vertex]])

# Define the SimpleFaultSurface using the RectangularMesh
fault_mesh_chi_chi = RectangularMesh(lons, lats, depths)
rupture_surface_chi_chi = SimpleFaultSurface(fault_mesh_chi_chi)

The remaining path parameters can be obtained using the following methods of the `SimpleFaultSurface` object:

In [ ]:
# Rrup, Rjb, and Rx distances
df_stations['r_rup'] = rupture_surface_chi_chi.get_min_distance(site_mesh)
df_stations['r_jb'] = rupture_surface_chi_chi.get_joyner_boore_distance(site_mesh)
df_stations['rx'] = rupture_surface_chi_chi.get_rx_distance(site_mesh)
#Ztor 
df_stations['Ztor'] = rupture_surface_chi_chi.get_top_edge_depth()
df_stations

## 5.2. Planar fault: 2022 $M_{W}6.9$ Taitung, Taiwan Earthquake

To exemplify a `PlanarSurface` object, we will use information about the [`2022 Taitung` earthquake](https://earthquake.usgs.gov/earthquakes/eventpage/us7000i90q/executive). The $M_W$ 6.9 strike-slip event occurred near the plate boundary between the Philippine Sea and Eurasian plates at the southeast coast of Taiwan. Its main characteristics were retrieved from [USGS Earthquake Catalog (ComCat)](https://earthquake.usgs.gov/earthquakes/search/) and they are located at **data/Taitung_event.csv**:

In [ ]:
event_taitung = pd.read_csv("data/Taitung_event.csv")

Its `Finite Fault Model` was downloaded from USGS event's [ShakeMap](https://earthquake.usgs.gov/earthquakes/eventpage/usp0009eq0/shakemap/intensity) and can be find at **data/rupture_taitung.json**:

In [ ]:
rupture_taitung = gpd.read_file("data/rupture_taitung.json")

with open('data/rupture_taitung.json', 'r') as file:
    rupture_taitung_json = json.load(file)
rupture_taitung_json


We can compare it to the `1999 Chi-Chi` finite fault model using `pyGMT`. Note the difference in the rupture area between a $M_{W}7.6$ (3462 km<sup>2</sup>) and a $M_{W}6.9$ (672 km<sup>2</sup>) event. 

In [ ]:
fig = pygmt.Figure()

region = [119.5, 122.5, 21.5, 25.5]
proj = "T121/23.5/?"

with fig.subplot(nrows=1, ncols=2, figsize=["20c", "15c"], margins = "1c"):

    with fig.set_panel(panel=0):

        fig.grdimage(grid=grid, region=region, projection=proj, cmap="data/eleva.cpt")
        fig.basemap(frame=["a2f1g1", '+tMw7.6 Chi-Chi'])
        fig.plot(data=rupture, pen="2p,black", fill = "grey@40")
        fig.meca(spec=event.iloc[0,0:].to_dict(), scale="0.5c+f10p", labelbox="white@30", offset="+p1p,black+s0.15c")

    with fig.set_panel(panel=1):

        fig.grdimage(grid=grid, region=region, projection=proj, cmap="data/eleva.cpt")
        fig.basemap(frame=["a2f1g1", '+tMw6.9 Taitung'])
        fig.plot(data=rupture_taitung, pen="2p,black", fill = "grey@40")
        fig.meca(spec=event_taitung.iloc[0,0:].to_dict(), scale="0.5c+f10p", labelbox="white@30", offset="+p1p,black+s0.15c")

fig.show(width=1200)

We can define a `PlanarSurface()` using the coordinates of the four corner points in the following order: top-left, top-right, bottom-right, bottom-left. The azimuth of the line connecting the top-left and the top-right corners define the surface strike, while the angle between the line connecting the top-left and bottom-left corners and a line parallel to the Earth surface defines the surface dip:

<img src="fig/def_PlanarSurface.jpg" alt="Image 2" width="400"><br>

In [ ]:
polygon_coordinates = rupture_taitung_json['features'][0]['geometry']['coordinates'][0][0]

points = []
for lon, lat, depth in polygon_coordinates[:-1]:  # Exclude last point because it close the polygon
    points.append(Point(lon, lat, depth))
rupture_surface_taitung = PlanarSurface.from_corner_points(points[0], points[1], points[2], points[3])

Once the `PlanarSurface()` is defined, we can use the same methods as in the `SimpleFaultSurface()`:

In [ ]:
print(rupture_surface_taitung.get_min_distance(site_mesh))

# 6. Site parameters

The site parameters required to apply the `BSSA14` and `CY14` GMMs are:

* Travel-time averaged shear-wave velocity of the top 30 m ($V_{S30}$) 
* Depth to shear-wave velocity of 1.0 km/s ($Z_{1.0}$) - Basin parameter

These can be obtained from the **NGA-West2 flatfiles**:

In [ ]:
# Site parameters
df_stations['Vs30'] = df_chichi['Vs30 (m/s) selected for analysis'].values
df_stations['Z1'] = df_chichi['Northern CA/Southern CA - H11 Z1 (m)'].values
df_stations

Note that in **NGA-West2** database, `-999` is a numeric placeholder to indicate a **missing value** (`N/A`). This is a common issue in the application of GMMs since detailed site characterization of seismic stations is limited worldwide. In fact, only `20.8%` of the NGA‐West2 seismic stations have measured have measured $V_{S30}$ and only `10%` possess $V_S$ profiles that penetrate deep enough to define the $Z_{1.0}$ velocity horizon. For our database:

In [ ]:
df_stations = df_stations.replace(-999, np.nan)
df_stations.info()

Although some seismic stations do not have measured $V_{S30}$, **NGA-West2 flatfile** provides a recommended value for all the stations. For `Taiwan`, only `300` of the 420 seismic stations had measured $V_{S30}$. In the absence of measurements, the recommended values were **inferred using geotechnical, geology, and/or topographic slope proxies**. Some of these models were developed for specific regions (*i.e.,*  [Willis and Clahan 2006](https://pubs.geoscienceworld.org/ssa/bssa/article-abstract/96/4A/1483/146717/Developing-a-Map-of-Geologically-Defined-Site?redirectedFrom=fulltext) for California or [Matsuoka *et al.* 2006](https://www.jstage.jst.go.jp/article/jsceseee/23/1/23_1_57s/_article) for Japan); however, most could be applied globally (*i.e.,* [Wald and Allen 2007](https://pubs.geoscienceworld.org/ssa/bssa/article-abstract/97/5/1379/146527/Topographic-Slope-as-a-Proxy-for-Seismic-Site?redirectedFrom=fulltext)). More recently, USGS has developed a `global hybrid` $V_{S30}$ map [(Heath *et al.*, 2020)](https://journals.sagepub.com/doi/10.1177/8755293020911137) based on topographic slope but incorporating regional maps for certain regions. This map can be accessed using an [online map interface](https://apps.usgs.gov/earthquakes/vs30/). For more details about the $V_{S30}$ estimation in the  **NGA-West2 database**, check [Seyhan *et al.* (2014)](https://onlinelibrary-wiley-com.prox.lib.ncsu.edu/doi/full/10.1193/062913EQS180M?sid=vendor%3Adatabase).



On the other hand, inferred values for the basin parameters (*i.e.*, $Z_{1.0}$) were only reported for California and Japan. These values were obtained based on regional 3D velocity models. As a side note, the Geohazards and Earthquake Eng. Research Team (`GEOQUAKE`) at NCSU has recently developed a suite of regional velocity models for the US Atlantic and Coastal Plains ([Gann-Phillips *et al.*, 2024](https://journals.sagepub.com/doi/full/10.1177/87552930231222960); [Gann-Phillips *et al.*, 2026](https://onlinelibrary.wiley.com/doi/10.1002/esp4.70114)). 

For our application, we can estimate $Z_{1.0}$ based on $V_{S30}$ using the relationship for non-Japan regions proposed by [Chiou and Youngs (2014)](https://journals.sagepub.com/doi/10.1193/072813EQS219M):

$$\ln(Z_{1.0}) = -\frac{7.15}{4} \ln\left(\frac{V_{S30}^4 + 571^4}{1360^4+571^4}\right)$$

In [ ]:
df_stations["Z1"] = np.where(df_stations["Z1"].isna(), np.round(np.exp(-7.15/4*np.log((df_stations["Vs30"]**4+571**4)/(1360**4+571**4))),1), df_stations["Z1"])
df_stations.head()

# 7. Predictions using GMMs

**OpneQuake Engine** has one of the largest and most comprehensive open-source compilations of GMMs in the world. **OpenQuake Engine** features more than **150 models** for different tectonic regimes, including active tectonic regions (*e.g.*, **NGA-West2**), stable continental regions (*e.g.*, **NGA-East**), subduction regions (*e.g.*, **NGA-Sub**), and geothermal areas. Models are actively audited by seismologists, and new GMMs are constantly added to the library. This is an important feature as the number of GMMs has increased exponentially in last decades (for a summary of the GMMs published from 1964 to 2023, see [Douglas, 2024](https://figshare.com/articles/online_resource/Ground_motion_prediction_equations_1964-2023_incomplete_/26827750?file=48769036)). The list of available GMMs can be found on its [website](https://docs.openquake.org/oq-engine/3.20/manual/user-guide/inputs/ground-motion-models-inputs.html):

In [ ]:
from openquake.hazardlib import gsim

all_models = gsim.get_available_gsims()

for model_name in sorted(all_models.keys()):
    print(model_name)

To use a specific **GMM** in `Openquake.Engine`, we need to:

* Select a valid GMM (**`gsim`**) from the library.
* Create a list of IM types (**``imt``**) of interest. The functions include:
    * Peak Ground Acceleration (g): `PGA()` 
    * Peak Ground Velocity (cm/s $^2$): `PGV()` 
    * Pseudo-spectral acceleration (g): `SA()`
* Create a **``ContextMaker``** object. A class that encloses all the rupture, path, and site parameters (contexts) needed to apply the GMM.

## 7.1. BSSA14

The code for  `BSSA14` GMM is  [`BooreEtAl2014`](https://docs.openquake.org/oq-engine/3.20/manual/api-reference/openquake.hazardlib.gsim.html#module-openquake.hazardlib.gsim.boore_2014).


In [ ]:
from openquake.hazardlib.gsim.boore_2014 import BooreEtAl2014
from openquake.hazardlib.imt import PGA, PGV, SA
from openquake.hazardlib.contexts import simple_cmaker


We will estimate PGA, PGV, and PSA for 10 periods (0.02, 0.04, 0.08, 0.1, 0.2, 0.4, 0.8, 1.0, 2.0, and 3.0 s):

In [ ]:
# GMM
gsim = BooreEtAl2014()

# Define IMTs 
imts = [PGA(), PGV()]

periods = [0.02,0.04,0.08,0.1,0.2,0.4,0.8,1,2,3]
for T in periods:
    imts.append(SA(T))
imt_strings = [str(imt) for imt in imts]
print(imt_strings)

The `ContextMaker` is defined by the `simple_cmaker()` function. First, we have to define the GMM, the IMs, and the magnitude of our event. Then, we have to create a `ctx()` object, which consists of a `numpy.array` whose rows correspond to the number of target sites and whose columns are the input parameters. For `BSSA14` the input parameters are $M_W$, $\lambda$, $R_{JB}$, and $V_{S30}$:

In [ ]:
n = len(df_stations)
mag = np.full(n,event['magnitude'][0])

# Setup Context Maker for ALL IMTs
cmaker = simple_cmaker(
    [gsim], imt_strings, mags=["%2.2f" % m for m in mag]
)

ctx = cmaker.new_ctx(n)
# Source parameters
ctx["mag"] = mag 
ctx["rake"] = np.full(n, event['rake'][0])
# Path parameters
ctx["rjb"] = df_stations['r_jb']
# Site parameters
ctx["vs30"] = df_stations['Vs30']
print(ctx[:10])


To compute the mean and standard deviation of the predictions, we use the `get_mean_stds()` function of the `ContextMaker` object:

In [ ]:
# Run OpenQuake calculation
means, sigmas, taus, phis = cmaker.get_mean_stds([ctx])


We can sort the results into a `pandas.DataFrame` format:

In [ ]:
# Extract results for each IMT
predicted_data = []

for imt_idx, imt_str in enumerate(imt_strings):
    # Extract arrays for the first GSIM (index 0) and current IMT
    mean_val = means[0][imt_idx]
    sigma_tot = sigmas[0][imt_idx]
    tau = taus[0][imt_idx]
    phi = phis[0][imt_idx]
    
    # Store results row-by-row or append to a structural dataframe
    df_imt = pd.DataFrame({
        'station': df_stations['station'],
        'rjb': df_stations['r_jb'],
        'Vs30': df_stations['Vs30'],
        'imt': imt_str,
        'mean_log': mean_val,
        'sigma': sigma_tot,
        'tau': tau,
        'phi': phi
    })
    predicted_data.append(df_imt)

# Combine all IMT data into a single DataFrame
df_BSSA14 = pd.concat(predicted_data, ignore_index=True)
df_BSSA14.to_csv('data/result/BSSA14.csv', index=False)
print(df_BSSA14.head())

In modern GMMs, it is common to partition the total variability of the model ($\sigma_{\log IM}$) into between- ($ \delta B_{i}$) and within-event ($\delta W_{i,j}$) variability:

$$\varepsilon_{ij} = \delta B_{i} + \delta W_{i,j}$$

with:

$$\delta B \sim \mathcal{N}\left(0,\tau\right) ;\quad  \delta W \sim \mathcal{N}\left(0,\phi \right)$$

According to [(Al Atik *et al.*, 2010)](https://pubs.geoscienceworld.org/ssa/srl/article/81/5/794/143735/The-Variability-of-Ground-Motion-Prediction-Models), $\delta B_{i}$ represents:

* Average shift of the observed GMs from the population median predicted by the GMM 
* Average source effects that are not capture in the GMM

and $\delta W_{i,j}$ represents:

* Azimuthal variations in source, path, and site effects
* Influences of factors not accounted by path and site parameters

<img src="fig/Al_Atik_2010_Between_Within_Residuals.png" alt="Image 2" width="500"><br>
<small style="color: #666;">Image credit: <a href="https://pubs.geoscienceworld.org/ssa/srl/article/81/5/794/143735/The-Variability-of-Ground-Motion-Prediction-Models" target="_blank">Al Atik et al. (2010)</a></small>

Note that for our case, $\phi$ is not constant. `BSSA14`'s intraevent residuals include dependence on magnitude and the degree of nonlinear soil response.

## 7.2. CY14

The code for `CY14` GMM is [`ChiouYoungs2014`](https://docs.openquake.org/oq-engine/3.20/manual/api-reference/openquake.hazardlib.gsim.html#module-openquake.hazardlib.gsim.chiou_youngs_2014):

In [ ]:
from openquake.hazardlib.gsim.chiou_youngs_2014 import ChiouYoungs2014

 For `BSSA14` the input parameters are $M_W$, $\delta$, $\lambda$, $Z_{tor}$, $R_{JB}$, $R_{rup}$, $R_{X}$, $V_{S30}$, and $Z_{1.0}$. Thus, to definte the `ContextMaker`:

In [ ]:
# GMM
gsim = ChiouYoungs2014()

# Setup Context Maker for ALL IMTs
cmaker = simple_cmaker(
    [gsim], imt_strings, mags=["%2.2f" % m for m in mag]
)

# Context maker
ctx2 = cmaker.new_ctx(n)
# Source parameters
ctx2["mag"] = mag 
ctx2["dip"] = np.full(n, event['dip'][0])
ctx2["rake"] = np.full(n, event['rake'][0])
ctx2["ztor"] = df_stations['Ztor']
# Path parameters
ctx2["rjb"] = df_stations['r_jb']
ctx2["rrup"] = df_stations["r_rup"]
ctx2["rx"] = df_stations["rx"]
# Site parameters
ctx2["vs30"] = df_stations['Vs30']
ctx2["z1pt0"] = df_stations['Z1']


To calculate the mean and standard deviations:

In [ ]:
# Run OpenQuake calculation
means, sigmas, taus, phis = cmaker.get_mean_stds([ctx2])

# Extract results for each IMT
predicted_data = []

for imt_idx, imt_str in enumerate(imt_strings):
    # Extract arrays for the first GSIM (index 0) and current IMT
    mean_val = means[0][imt_idx]
    sigma_tot = sigmas[0][imt_idx]
    tau = taus[0][imt_idx]
    phi = phis[0][imt_idx]
    
    # Store results row-by-row or append to a structural dataframe
    df_imt = pd.DataFrame({
        'station': df_stations['station'],
        'rrup': df_stations['r_rup'],
        'Vs30': df_stations['Vs30'],
        'imt': imt_str,
        'mean_log': mean_val,
        'sigma': sigma_tot,
        'tau': tau,
        'phi': phi
    })
    predicted_data.append(df_imt)

# Combine all IMT data into a single DataFrame
df_CY14 = pd.concat(predicted_data, ignore_index=True)
df_CY14.to_csv('data/result/CY14.csv', index=False)
print(df_CY14.head())

# 8. Comparison of model prediction with observed ground motions

We can evaluate a **GMM** performance by comparing the model predictions with observed ground motions (`residual analysis`). These comparisons are iteratively done during **GMMs** development, and they are a fundamental step in **GMM** selection for a specific region (*e.g.*, [McNamara *et al.* 2020](https://pubs.geoscienceworld.org/ssa/srl/article/91/1/183/579564/Evaluation-of-Ground-Motion-Models-for-U-S)). In this context, the mathematical representation of the GMM is reformulated as:

$$\log im_{ij} = \mu_{\log IM}(M_{i}, R_{ij}, S_{ij}) + \Delta_{ij}$$

where $\log im_{ij}$ is the natural logarithm of the observed IM from the i-th earthquake at the j-th site, $\mu_{\log IM}(M_{i}, R_{ij}, S_{ij})$ is the prediction of the **GMM**, $\Delta_{ij}$ is the total residual for the j-th site from the i-th earthquake. Furthermore, the total residual is partitioned into between-event residuals for the i-th earthquake ($\delta B_i$) and the within-event residual at the j-th site from the i-th earthquake ($\delta W_{ij}$). For simplicity, we will limit our evaluation to `total residuals`.

## 8.1. Residual calculation

We need to sort the observations from the **NGA-West2 flatfile** into a compatible format. First, we will extract the IMs of interest:

In [ ]:
cols_name = ["Station Name","PGA (g)", "PGV (cm/sec)", "T0.020S","T0.040S","T0.080S","T0.100S","T0.200S","T0.400S","T0.800S","T1.000S","T2.000S","T3.000S"]
observed_data = df_chichi[cols_name].reset_index(drop=True)
# Renaming the columns
observed_data.columns = ['station'] + np.unique(df_CY14['imt']).tolist()
observed_data.head()

Note that the database has some **missing values** (`-999`):

In [ ]:
observed_data = observed_data.replace(-999, np.nan)
print(observed_data.info())

Reconfiguring the dataset to match the format of our prior computations:

In [ ]:
# Melt data in format
obs_df = observed_data.melt(
    id_vars=['station'], 
    var_name='imt', 
    value_name='obs'
)
obs_df

Now, we can append our observations to the dataframes for `BSSA14` and `CY14`:

In [ ]:
df_BSSA14['obs_log'] = np.log(obs_df['obs'])
df_CY14['obs_log'] = np.log(obs_df['obs'])

# Remove NaN
df_BSSA14 = df_BSSA14.dropna().copy().reset_index(drop=True)
df_CY14 = df_CY14.dropna().copy().reset_index(drop=True)



We will omit all sites falling outside the applicability ranges of our GMMs:

In [ ]:
# Remove sites outside the applicability range
# BSSA14
BSSA14 = df_BSSA14[
    (df_BSSA14['rjb']<=400) &
    (df_BSSA14['Vs30'] >= 150) & 
    (df_BSSA14['Vs30'] <= 1500)
    ].copy().reset_index(drop=True)

# CY14
CY14 = df_CY14[
    (df_CY14['rrup']<=300) &
    (df_CY14['Vs30'] >= 180) & 
    (df_CY14['Vs30'] <= 1500)
    ].copy().reset_index(drop=True)

The total residuals are defined as:

$$\Delta_{ij} = \log im_{ij} - \mu_{\log IM}(M_{i}, R_{ij}, S_{ij})$$

In [ ]:
# Residuals
BSSA14['res_tot'] = BSSA14['mean_log'] - BSSA14['obs_log']
CY14['res_tot'] = CY14['mean_log'] - CY14['obs_log']
BSSA14.head()

## 8.2. Residual statistics and visualization

We can calculate the statistics (mean, standard deviation, min, max) of the residuals:

In [ ]:
# Statistics for each IM
stats_BSSA14 = BSSA14.groupby('imt')['res_tot'].agg(['count','mean','std','min','max']).reset_index()
stats_CY14 = CY14.groupby('imt')['res_tot'].agg(['count','mean','std','min','max']).reset_index()
print(stats_BSSA14)


From a `total residual` perspective, the best-performing GMMs are those with the mean closest to zero and the lowest total mean residual across all periods:

In [ ]:
import matplotlib.pyplot as plt

# BSSA14
plt.plot(periods, stats_BSSA14['mean'].iloc[2:], 
         color='red', marker='s', markersize=6, markeredgecolor='black', 
         markeredgewidth=0.5, linewidth=0.8, label='BSSA14', zorder=3)
plt.fill_between(periods, stats_BSSA14['mean'].iloc[2:] - stats_BSSA14['std'].iloc[2:], stats_BSSA14['mean'].iloc[2:] + stats_BSSA14['std'].iloc[2:],
        color='red', alpha=0.15, zorder=2)
# CY14
plt.plot(periods, stats_CY14['mean'].iloc[2:], 
         color='darkblue', marker='v', markersize=6, markeredgecolor='black',
         markeredgewidth=0.5, linewidth=0.8, label='CY14', zorder=3)
plt.fill_between(periods, stats_CY14['mean'].iloc[2:] - stats_CY14['std'].iloc[2:], stats_CY14['mean'].iloc[2:] + stats_CY14['std'].iloc[2:], 
                 color='darkblue', alpha=0.15, zorder=2
                 )

# General settings
plt.xscale('log')
plt.axhline(0, color='black', linestyle='--', linewidth=1.5)
plt.xlabel('Oscillator Period $T$ (s)')
plt.ylabel('Total Residual ($\Delta$)')
plt.legend(loc='upper right', frameon=False)
plt.show()

<div class="alert alert-block alert-info">
<b>Question:</b> What patterns can you recognize from the residual plot? Is GMMs performance uniform across all time periods?</br>
</div>